# 🌱 Washamba Bots — Learning to Farm

Welcome to our Kaggriculture journey.

We are **washamba_bots**, a team building an autonomous farmer for Kaggriculture.

Our first agent is called:

## `nikaangukia_meroni`

The name comes from the famous Kenyan "Nikaangukia Meroni" TikTok moment. 😂

We started with a deliberately simple deterministic agent and are using experiments to progressively make it smarter.

Rather than jumping straight into a complicated AI system, we are following an experimental process:

**Observe → Interpret → Decide → Act → Measure → Learn**

This notebook documents that process.

In [ ]:
## 🧑🏾‍🌾 What is Kaggriculture?

Kaggriculture is a two-player farming simulation.

Each farmer acts once per turn over a 30-day season, with 24 turns per day, giving a maximum of:

**30 × 24 = 720 turns**

The objective is simple:

> Have the most money in the bank at the end of the season.

But achieving that requires balancing several competing systems:

- 🌱 planting crops
- 💧 watering crops
- 🌾 harvesting
- 🐄 caring for animals
- 💰 buying seeds and other resources
- 🏪 selling products
- 📈 responding to a dynamic market
- 🧑🏾‍🌾 hiring farm hands
- 🗺️ expanding the farm
- 📦 managing shed capacity

Our challenge is to build an agent that can make these decisions autonomously.

In [ ]:
# Install the Kaggle environment package used by Kaggriculture.

!pip install -q --upgrade "kaggle-environments>=1.32.2"

In [ ]:
from kaggle_environments import make

env = make("kaggriculture", debug=True)

print("Environment:", env.name)
print("Version:", env.version)
print("Players:", env.specification.agents)
print("Maximum steps:", env.configuration.episodeSteps)

## 🔎 Understanding the Agent's Observation

Before building an agent, we need to understand what the environment tells it.

Each turn, the agent receives an observation containing information about:

### Public state

- `day`
- `hour`
- both players' farm states
- market prices
- market inventory
- unlocked town shops

### Private state

- our shed inventory
- our seeds
- inventories carried by the farmer and hired hands

The agent then returns actions such as:

```python
{
    "farmer": ["WATER"],
    "hands": [],
    "market": []
}

In [ ]:

# CELL 6 — Code

```python
# Run a game between two built-in random agents so we can inspect
# what the environment gives us.

env = make("kaggriculture", debug=True)

env.run(["random", "random"])

obs = env.steps[1][0].observation

print("Player:", obs.player)
print("Day:", obs.day)
print("Hour:", obs.hour)

print("\nUnlocked quadrants:")
print(obs.farms[obs.player].unlocked_quadrants)

print("\nMarket prices:")
for product, price in obs.market.prices.items():
    print(f"{product:12s} ${price}")

In [ ]:
farm = obs.farms[obs.player]
private = obs.private

print("=== FARM ===")
print("Money:", farm.money)
print("Farmer position:", farm.farmer)
print("Hands:", farm.hands)
print("Unlocked land:", farm.unlocked_quadrants)
print("Hires today:", farm.hires_today)

print("\n=== SEEDS ===")
for product, quantity in private.seeds.items():
    if quantity:
        print(product, quantity)

print("\n=== SHED ===")
for product, quantity in private.shed.items():
    if quantity:
        print(product, quantity)

# 🌱 Our First Agent

Our first agent is deliberately simple.

It is **deterministic** and does not use machine learning or an LLM.

Its basic decision hierarchy is:

1. Harvest a genuinely mature crop under the farmer.
2. Water a crop that needs watering.
3. Move toward an urgent task.
4. Plant when an appropriate empty tile is available.
5. Move toward the next useful tile.
6. Otherwise pass.

The first version also evaluates crops using their economics, including:

- price
- expected yield
- crop maturity time
- current market inventory

This gave us our first baseline:

## `nikaangukia_meroni`

In [ ]:
# Clone the Washamba Bots repository.

!git clone -q https://github.com/Kinjuriu/washamba_bots.git

In [ ]:
import sys
sys.path.append("/kaggle/working/washamba_bots")

from main import nikaangukia_meroni

print("Agent loaded successfully:", nikaangukia_meroni.__name__)

## 🧪 First Live Test

Before attempting to make our agent sophisticated, we want to establish a reproducible baseline.

We will first test:

**nikaangukia_meroni vs random**

The purpose of this experiment is not to win the competition.

It is to answer a much simpler question:

> Does our agent survive a complete 720-turn game without crashing?

In [ ]:
from kaggle_environments import make

env = make("kaggriculture", debug=True)

env.run([
    nikaangukia_meroni,
    "random"
])

final = env.steps[-1]

for i, player in enumerate(final):
    print(
        f"Player {i}: "
        f"reward={player.reward}, "
        f"status={player.status}"
    )

### Why this test matters

An agent that crashes, returns invalid actions, or gets stuck is not useful regardless of how clever its strategy looks.

Our first engineering gate is therefore:

> **Does it work?**

Only after that do we ask:

> **Is it good?**

In [ ]:
# Render the completed game.

env.render(
    mode="ipython",
    width=1000,
    height=700
)

# 🔬 Looking Inside the Agent

A final score tells us **what happened**.

It does not tell us **why**.

To understand our agent, we can inspect the sequence of actions it took during the game.

This allows us to ask questions such as:

- How often did it plant?
- How often did it water?
- How often did it harvest?
- How often did it sell?
- Did it spend too much time moving?
- Did it create more crops than one farmer could maintain?
- Did it accumulate products without selling them?

In [ ]:
from collections import Counter

farmer_actions = []

for step in env.steps:
    if not step:
        continue

    player = step[0]

    if not hasattr(player, "action"):
        continue

    action = player.action

    if isinstance(action, dict):
        farmer_action = action.get("farmer")

        if isinstance(farmer_action, list) and farmer_action:
            farmer_actions.append(farmer_action[0])
        elif isinstance(farmer_action, str):
            farmer_actions.append(farmer_action)

counts = Counter(farmer_actions)

print("Farmer action histogram:\n")

for action, count in counts.most_common():
    print(f"{action:10s} {count}")

In [ ]:
# Market action counts

market_actions = []

for step in env.steps:
    if not step:
        continue

    player = step[0]

    if not hasattr(player, "action"):
        continue

    action = player.action

    if isinstance(action, dict):
        for market_action in action.get("market", []):
            if isinstance(market_action, list) and market_action:
                market_actions.append(market_action[0])

print("Market actions:\n")

for action, count in Counter(market_actions).most_common():
    print(f"{action:15s} {count}")

# 🧩 First Major Finding: Activity Is Not the Same as Productivity

Our early experiments revealed an unexpected pattern.

The agent was highly active:

- many planting actions
- many watering actions
- many harvest actions
- frequent movement

However, activity did not necessarily translate into money.

In one diagnostic run, our agent:

| Action | Our agent | Starter |
|---|---:|---:|
| HARVEST | 237 | 9 |
| WATER | 122 | 30 |
| PLANT | 23 | 10 |
| PASS | 118 | 671 |
| SELL orders | 4 | 9 |
| End-game weeds | 22 | 3 |

This raised an important question:

> **Can a single farmer maintain everything that our decision logic asks it to plant?**

The answer appears to be no.

The agent can create work faster than it can perform the required maintenance.

This gives us a new concept for agent design:

## Resource-constrained planning

A good action is not simply an action that is locally profitable.

It must also be **maintainable with the resources available to the agent.**

In [ ]:
# Inspect the final farm state.

final_obs = env.steps[-1][0].observation
final_farm = final_obs.farms[final_obs.player]

tile_counts = Counter()

for row in final_farm.tiles:
    for tile in row:
        if tile is None:
            tile_counts["EMPTY"] += 1
        elif tile == "LOCKED":
            tile_counts["LOCKED"] += 1
        elif isinstance(tile, dict):
            tile_counts[tile.get("kind", "UNKNOWN")] += 1

print("Final farm footprint:\n")

for tile_type, count in tile_counts.items():
    print(f"{tile_type:10s} {count}")

In [ ]:
import matplotlib.pyplot as plt

labels = list(counts.keys())
values = list(counts.values())

plt.figure(figsize=(10, 5))
plt.bar(labels, values)

plt.title("nikaangukia_meroni — Farmer Actions")
plt.xlabel("Action")
plt.ylabel("Number of turns")

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# 🧠 From Reactive Rules to Planning

Our first agent was primarily reactive.

It asked:

> "What should I do right now?"

But farming introduces an important constraint:

> **Today's decisions create tomorrow's work.**

Planting a crop does not only create a future harvest.

It also creates future watering obligations.

Therefore:

**Planting more ≠ automatically earning more.**

A better agent needs to reason about the future workload it creates.

For example:

```text
PLANT
  ↓
crop becomes a future obligation
  ↓
WATER every day
  ↓
WAIT for maturity
  ↓
HARVEST
  ↓
STORE
  ↓
SELL


---

# 12. And THIS is where our next agent begins

I want you to put this at the end for now:

```markdown
# 🚜 What Comes Next?

Our first deterministic agent has taught us something more valuable than a score.

We need to move from:

**"Which crop should I plant?"**

toward:

**"What farming plan can I actually execute?"**

Our next experiments will investigate:

1. Crop workload and watering capacity
2. When planting should be delayed
3. Whether hiring farm hands changes the optimal strategy
4. When fertilizer is economically worthwhile
5. Shed capacity and selling decisions
6. Land expansion
7. Market supply and demand
8. Opponent-aware strategy

Only after establishing these deterministic foundations will we investigate whether more complex agent architectures, including LLM-assisted planning, can improve performance.